# Cleaning the dataset


Variable overview
* Raw_df: the unchanged dataframe
* df: the dataframe wiht irrelevant columns dropped

## Load dependencies and Dataset

In [126]:
import pandas as pd
import chardet
import random
import textwrap
from ftfy import fix_text
import re
from collections import Counter


In [127]:
path = "data/stylecom_raw.csv"


In [128]:
raw_df = pd.read_csv(path, encoding="latin-1")


raw_df.shape
raw_df.info()
raw_df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6629 entries, 0 to 6628
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   year        6629 non-null   int64 
 1   season      6629 non-null   object
 2   designer    6629 non-null   object
 3   author      6492 non-null   object
 4   city        6629 non-null   object
 5   date        6629 non-null   object
 6   image       6629 non-null   object
 7   review      6629 non-null   object
 8   Unnamed: 8  5 non-null      object
dtypes: int64(1), object(8)
memory usage: 466.2+ KB


year             0
season           0
designer         0
author         137
city             0
date             0
image            0
review           0
Unnamed: 8    6624
dtype: int64

In [129]:
raw_df.head()

,year,season,designer,author,city,date,image,review,Unnamed: 8
0,2000,Spring,Matt Nye,Armand Limnander,NEW YORK,17-Sep-99,"<div class=""image-container""><a class=""slidesh...",Designer Matt Nye's sophomore show featured a ...,NaN
1,2000,Spring,Giorgio Armani,Armand Limnander,MILAN,29-Sep-99,"<div class=""image-container""><a class=""slidesh...","Armani proposed a light, feminine silhouette f...",NaN
2,2000,Spring,Eric Bergre,Armand Limnander,PARIS,4-Oct-99,"<div class=""image-container""><a class=""slidesh...",Broadway Garnier was the theme for Eric Berg?r...,NaN
3,2000,Spring,C_line,Armand Limnander,PARIS,7-Oct-99,"<div class=""image-container""><a class=""slidesh...",Getaway glamour was the theme for Celine's str...,NaN
4,2000,Spring,Byblos,Armand Limnander,MILAN,27-Sep-99,"<div class=""image-container""><a class=""slidesh...",Judo Jetson blends my favorite cartoon charact...,NaN


## Cleaning

### Dropping uninformative columns

In [130]:
df = raw_df.drop(columns=["Unnamed: 8", "image"], axis=1)
df.head()

,year,season,designer,author,city,date,review
0,2000,Spring,Matt Nye,Armand Limnander,NEW YORK,17-Sep-99,Designer Matt Nye's sophomore show featured a ...
1,2000,Spring,Giorgio Armani,Armand Limnander,MILAN,29-Sep-99,"Armani proposed a light, feminine silhouette f..."
2,2000,Spring,Eric Bergre,Armand Limnander,PARIS,4-Oct-99,Broadway Garnier was the theme for Eric Berg?r...
3,2000,Spring,C_line,Armand Limnander,PARIS,7-Oct-99,Getaway glamour was the theme for Celine's str...
4,2000,Spring,Byblos,Armand Limnander,MILAN,27-Sep-99,Judo Jetson blends my favorite cartoon charact...


### Helper functions

In [131]:
def view_reviews(df, column):
    for text in df[column].sample(5):
        print(textwrap.fill(text, width=80))
        print("-" * 80)

view_reviews(df, "review")

Roksanda Ilincic is a strikingly tall, slender Serbian with a propensity, as a
designer, for exaggerating things. In the past, if she'd done a pompom or a
taffeta bow, it turned ginormous, and her flounces and explosions of tulle could
go on into the next week. "I can't help myself once I start playing," she said.
"And this time, I've tried to make the dresses look a bit like fashion sketches
from the thirties to the seventies from a book I got for my birthday." Thus,
when her lovely long slivers of YSL-ish charmeuse occasionally encountered a
jacket, the outcome was a supersize inflation in the shoulder department. There
were two kimono-type affairs and a "smoking" bolero that was twice as wide as it
was long. As jokey as this sounds, the collection was imbued with Ilincic's
innate elegance; the models somehow even managed to look like the designer at a
party. This season, there was less of the raw edge and more sophistication, as
in a long-sleeved ivory wrap with the shoulder line pi

### Checking broken character encoding

In [132]:
TEXT_COLS = ["designer", "review"]


def suspicious_tokens(text):
    """
    Find all tokens containing ?, _, or C1/control-byte artifacts from latin1 read.
    """
    if not isinstance(text, str):
        return []
    # catches tokens containing ?, _, or C1/control-byte artifacts from latin1 read
    return re.findall(r"\b\S*[?_-]\S*\b", text)

def count_suspicious_tokens(df, text_cols):
    """
    Count suspicious tokens across specified text columns in the DataFrame.
    """
    token_counts = Counter()
    for col in text_cols:
        for text in df[col].dropna():
            token_counts.update(suspicious_tokens(text))

    suspect_df = (
        pd.DataFrame(token_counts.items(), columns=["token", "count"])
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    return suspect_df

suspect_df = count_suspicious_tokens(df, TEXT_COLS)
suspect_df.head(200)

suspect_df.to_csv("suspect_tokens.csv", index=False)
print(len(suspect_df))

10347


## Fixing broken character encoding

In [133]:
# General fixing of text encoding issues
df["review_fixed"] = df["review"].apply(fix_text)

### "Reviews" column

In [136]:
# Dictionaries for replacements
name_replacements = {
    r"(?<!\w)Berg\?re(?!\w)": "Bergère",
    r"(?<!\w)Valer\?e(?!\w)": "Valérie",
    r"(?<!\w)Chlo\?(?!\w)": "Chloé",
    r"(?<!\w)C[\?_]line(?!\w)": "Céline",
    r"(?<!\w)Ferr\?(?!\w)": "Ferré",
    r"(?<!\w)Abaet\?(?!\w)": "Abaeté",
    r"(?<!\w)R\?yes(?!\w)": "Reyes",
    r"(?<!\w)Azrou\?l(?!\w)": "Azrouël",
    r"(?<!\w)K\?hne(?!\w)": "Kühne",
    r"(?<!\w)St\?rk(?!\w)": "Stärk",
    r"(?<!\w)Comme des Gar\?ons(?!\w)": "Comme des Garçons",
    r"(?<!\w)Herm\?s(?!\w)": "Hermès",
    r"(?<!\w)Herv\? L\?ger(?!\w)": "Hervé Léger",
    r"(?<!\w)V\?ronique(?!\w)": "Véronique",
    r"(?<!\w)D\?tacher(?!\w)": "Détacher",
    r"(?<!\w)Rhi\?(?!\w)": "Rhié",
    r"(?<!\w)Mus\?e(?!\w)": "Musée",
    r"Beyonc\?": "Beyoncé",
}

contraction_replacements = {
    r"(?<=\w)n\?t(?!\w)": "n't",
    r"(?<=\w)\?re(?!\w)": "'re",
    r"(?<=\w)\?ve(?!\w)": "'ve",
    r"(?<=\w)\?ll(?!\w)": "'ll",
    r"(?<=\w)\?d(?!\w)": "'d",
    r"(?<=\w)\?m(?!\w)": "'m",
    r"(?<=\w)\?s(?!\w)": "'s",
}

domain_lanugage_replacements = {
    r"(?<!\w)neglig\?e(?!\w)": "négligée",
    r"(?<!\w)d\?collet\?(?!\w)": "décolleté",
    r"(?<!\w)d\?collet\?s(?!\w)": "décolletés",
    r"(?<!\w)d\?collet\?e(?!\w)": "décolletée",
    r"(?<!\w)lingerie(?!\w)": "lingerie",
    r"(?<!\w)boucl\?(?!\w)": "bouclé",
    r"(?<!\w)piqu\?(?!\w)": "piqué",
    r"(?<!\w)lam\?(?!\w)": "lamé",
    r"(?<!\w)cr\?pe(?!\w)": "crêpe",
    r"(?<!\w)velour(?!\w)": "velour",
    r"(?<!\w)velours(?!\w)": "velours",
    r"(?<!\w)appliqu\?(?!\w)": "appliqué",
    r"(?<!\w)appliqu\?s(?!\w)": "appliqués",
    r"(?<!\w)pliss\?(?!\w)": "plissé",
    r"(?<!\w)pliss\?e(?!\w)": "plissée",
    r"(?<!\w)pliss\?s(?!\w)": "plissés",
    r"(?<!\w)matelass\?(?!\w)": "matelassé",
    r"(?<!\w)prot\?g\?(?!\w)": "protégé",
    r"(?<!\w)prot\?g\?e(?!\w)": "protégée",
    r"(?<!\w)cach\?(?!\w)": "caché",
    r"(?<!\w)cach\?e(?!\w)": "cachée",
    r"(?<!\w)haute coutur\?(?!\w)": "haute couture",
    r"(?<!\w)pr\?t-\?-porter(?!\w)": "prêt-à-porter",
    r"(?<!\w)touch\?(?!\w)": "touché",
    r"(?<!\w)entr\?e(?!\w)": "entrée",
    r"(?<!\w)appliqu[?éèêë]e(?!\w)": "appliquée",
    r"(?<!\w)appliqu[?éèêë]es?(?!\w)": "appliquées",
    r"(?<!\w)d[?éèêë]collet[?éèêë](?!\w)": "décolleté",
    r"(?<!\w)d[?éèêë]collet[?éèêë]s(?!\w)": "décolletés",
}


In [139]:

def apply_replacements(df, replacements):
    df = df.copy()
    df["review_replaced"] = df["review_fixed"]

    for pattern, repl in replacements.items():
        df["review_replaced"] = df["review_replaced"].str.replace(
            pattern,
            repl,
            regex=True,
            case=False
        )

    return df


# Fix names and contractions first
replacements = name_replacements | contraction_replacements
df_replaced = apply_replacements(df, replacements)

#Then domain language
df_replaced_double = apply_replacements(df_replaced, domain_lanugage_replacements)

In [141]:
# View reviews
view_reviews(df_replaced_double, "review_replaced")

There are some designers who torture themselves to produce their best work, some
whose ideas flare defiantly under pressure, and some whose brilliance can only
flourish when they are set free from angst. Christian Lacroix belongs to the
last category: a creator motivated by happiness. That emotion?painted in a
fantastic kaleidoscope of color and sketched with a real idea of how a great
summer ought to feel to a girl?came through in every outfit that trod blissfully
along the bright yellow sand of his runway. In a season of print and femininity,
Lacroix put together one of the most ravishingly individual looks to be found
anywhere. Hair tied up in printed headscarves, his girls wore flowered dresses
with contrasting slubby coats or jackets slung on top. They walked lightly, in
Hessian ballet shoes, swinging straw bags, each decorated with a big satin bow
with a chunk of diamant? in the center. And within that template, he found ways
to play out all the experience of his nearly 20 years 

In [143]:
TEXT_COLS = ["review_replaced"]

remaining_suspect_df = count_suspicious_tokens(df_replaced_double, TEXT_COLS)
remaining_suspect_df.head(200)

remaining_suspect_df.to_csv("remaining_suspect_tokens.csv", index=False)
print(len(remaining_suspect_df))

10304


In [ ]:
df_replaced[["review", "review_fixed", "review_replaced"]].sample(10)

view_reviews(df_replaced, "review_replaced")

In [ ]:
pattern = r"(?<=\w)\?s(?!\w)"

unique_terms = (
    df_replaced_double["review_replaced"]
    .str.findall(pattern)
    .explode()
    .dropna()
    .unique()
    .tolist()
)

df_uterms = pd.DataFrame({"term": unique_terms})

df_uterms.to_csv("data/unique_terms.csv", index=False)


In [145]:
df_replaced_double[df_replaced_double["review_replaced"].str.contains(r"\?s", na=False)]["review_replaced"].head(10)

for x in df_replaced_double[df_replaced_double["review_replaced"].str.contains(r"\?s", na=False)]["review_replaced"].head(5):
    print(repr(x))

"Marni's presentation relied on youthful and easy-to-wear dresses and separates in bright pink, green and yellow over white and beige. Polka dots and abstract motifs in light cotton dominated, but there were also flowing country-girl skirts with subtle embroidery and traditional gingham prints. The translucent layering of light fabrics added a playful touch to the feminine silhouette. The look was completed with flat shoes and boots?some of which were also decorated with eye-catching polka dots."
"Tommy Hilfiger appropriated the current elements of decoration in fashion?beading, assorted prints, and embroidery?stripped them of all frilliness, and used them to rock the house at Madison Square Garden. Bush performed live as a parade of sexy modern-day cowboys and girls took to the runway wearing multi colored eagle bustiers, crimson motorcycle jackets, and jeans with snakeskin appliques. Red, white, and blue was the patriotic, and predominant, color scheme, occasionally tempered by patch

## Designer column

In [ ]:
designer_replacements = {
    "Chlo_": "Chloé",
    "C_line": "Céline",
    "Ferr_": "Ferré",
    "Abaet_": "Abaeté",
    "R?yes": "Reyes",
    "Yigal AzrouФl": "Yigal Azrouël",
    "Kai KЩhne": "Kai Kühne",
    "St_rk": "Stärk",
    "Comme des GarЌons": "Comme des Garçons",
    "HermЏs": "Hermès",
    "Herv_ L_ger by Max Azria": "Hervé Léger by Max Azria",
    "V?ronique Leroy": "Véronique Leroy",
    "A D_tacher": "A Détacher",
    "Rhi_": "Rhié"
}